In [0]:
%pip install openai

In [0]:
#COMMAND ----------
# MAGIC %md ## ⚙️ Configuration
# MAGIC Fill in your credentials below (or store in Databricks Secrets for production).

# ---------- USER CONFIG — edit these ----------
AZURE_AI_ENDPOINT  = ""  # Azure AI Foundry base_url (with trailing slash)
AZURE_AI_KEY=""                                                # Azure AI API key
DEPLOYMENT_NAME    = "DeepSeek-V3.2-Speciale"                                             # Your deployed model name

DATABRICKS_HOST    = ""
DATABRICKS_TOKEN   = ""                   # Personal Access Token

NOTEBOOK_FOLDER    = "/pipeline"                   # Where notebooks get created
JOB_NAME           = "POCPipeline_customer_analytics"
# ----------------------------------------------

# COMMAND ----------

import ast, json, base64, time, requests
from openai import OpenAI
from pyspark.sql import Row

client = OpenAI(base_url=AZURE_AI_ENDPOINT, api_key=AZURE_AI_KEY)
db     = {"Authorization": f"Bearer {DATABRICKS_TOKEN}", "Content-Type": "application/json"}

SYSTEM_PROMPT = (
    "You are a PySpark code generator. Output ONLY valid Python/PySpark code. "
    "No English sentences, no prose, no markdown fences, no preamble, no postamble. "
    "Every line must be valid Python. Start directly with import statements."
)

def _extract_code(raw):
    """Pull code out of markdown fences, then surgically remove any lines causing SyntaxErrors."""
    out = raw.strip()
    # 1. Pull out fenced block if present
    if "```" in out:
        parts = out.split("```")
        out = parts[1].strip()
        if out.startswith("python"): out = out[6:].lstrip("\n")
        elif out.startswith("py\n"): out = out[3:]
    # 2. Iteratively remove the exact line reported by each SyntaxError
    lines = out.splitlines()
    for _ in range(20):                          # max 20 bad lines to remove
        try:
            ast.parse("\n".join(lines))
            break
        except SyntaxError as e:
            bad = (e.lineno or 1) - 1            # 0-based index
            if 0 <= bad < len(lines):
                removed = lines.pop(bad)
                print(f"   🔧 Removed non-Python line {bad+1}: {removed[:80]!r}")
            else:
                break
    return "\n".join(lines).strip()

def ask_agent(system, user):
    for attempt in range(3):
        r = client.chat.completions.create(
            model=DEPLOYMENT_NAME,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            temperature=0.0 if attempt > 0 else 0.2,
        )
        out = _extract_code(r.choices[0].message.content)
        try:
            ast.parse(out)   # validate Python syntax
            return out
        except SyntaxError as e:
            print(f"⚠️  Attempt {attempt+1}/3 syntax error: {e} — retrying at temperature=0.0")
    return out  # return best effort after 3 attempts

# COMMAND ----------
# MAGIC %md ### Step 1 — Pipeline blueprint (agent designs code, structure is fixed)

# Blueprint is fixed — agent's job is to write the PySpark code, not redesign the structure
blueprint = {
    "pipeline_name": "Customer Analytics Pipeline",
    "stages": [
        {"stage_number": 1, "stage_name": "ingest",          "output_table": "stg_customers"},
        {"stage_number": 2, "stage_name": "preprocess",      "output_table": "stg_customers_clean"},
        {"stage_number": 3, "stage_name": "analyze",         "output_table": "temp_views"},
        {"stage_number": 4, "stage_name": "persist_results", "output_table": "results_summary_stats"},
    ]
}
print(f"✅ Blueprint: {blueprint['pipeline_name']}")
for s in blueprint["stages"]: print(f"   {s['stage_number']}. {s['stage_name']} → {s['output_table']}")

# COMMAND ----------
# MAGIC %md ### Step 2 — Agent writes PySpark code per stage

STAGE_PROMPTS = [
    "Write runnable Databricks PySpark: read samples.bakehouse.sales_customers via spark.table(), DROP TABLE IF EXISTS stg_customers, write to stg_customers Delta, print count.",
    "Write runnable Databricks PySpark: read stg_customers (columns: customerID, first_name, last_name, email_address, phone_number). Trim all string columns, lowercase email_address column (NOT 'email'). Add full_name=concat(first_name+space+last_name), email_domain=split(email_address,'@')[1], phone_digits=regexp_replace(phone_number,'[^0-9]',''), area_code=first 3 chars of phone_digits if len>=10. Filter where customerID isNotNull AND email_address isNotNull. DROP TABLE IF EXISTS stg_customers_clean, write to stg_customers_clean Delta. Use column name email_address throughout, never 'email'. Use column name phone_number throughout, never 'phone'.",
    """Write runnable Databricks PySpark. Import pyspark.sql.functions as F. Read df=spark.table('stg_customers_clean'). Columns available: customerID, first_name, last_name, email_address, phone_number, full_name, email_domain, phone_digits, area_code. Do NOT split or re-derive any column. Step 1: df1=df.groupBy('email_domain').count().orderBy(F.desc('count')).limit(20). spark.sql('DROP TABLE IF EXISTS results_top_email_domains'). df1.write.format('delta').mode('overwrite').saveAsTable('results_top_email_domains'). display(df1). Step 2: df2=df.groupBy('first_name').count().orderBy(F.desc('count')).limit(20). spark.sql('DROP TABLE IF EXISTS results_top_first_names'). df2.write.format('delta').mode('overwrite').saveAsTable('results_top_first_names'). display(df2). Step 3: df3=df.filter(F.col('area_code').isNotNull()).groupBy('area_code').count().orderBy(F.desc('count')).limit(20). spark.sql('DROP TABLE IF EXISTS results_top_area_codes'). df3.write.format('delta').mode('overwrite').saveAsTable('results_top_area_codes'). display(df3). Print done.""",
    """Write runnable Databricks PySpark. Import pyspark.sql.functions as F and datetime. Step 1: df=spark.table('stg_customers_clean'). total_customers=df.count(). unique_email_domains=df.select('email_domain').distinct().count(). unique_first_names=df.select('first_name').distinct().count(). Step 2: top_email_domain=spark.table('results_top_email_domains').orderBy(F.desc('count')).limit(1).collect()[0]['email_domain']. Step 3: top_first_name=spark.table('results_top_first_names').orderBy(F.desc('count')).limit(1).collect()[0]['first_name']. Step 4: from pyspark.sql import Row. summary=spark.createDataFrame([Row(total_customers=int(total_customers), unique_email_domains=int(unique_email_domains), unique_first_names=int(unique_first_names), top_email_domain=str(top_email_domain), top_first_name=str(top_first_name), pipeline_run_timestamp=str(datetime.datetime.now()))]). spark.sql('DROP TABLE IF EXISTS results_summary_stats'). summary.write.format('delta').mode('overwrite').saveAsTable('results_summary_stats'). display(summary).""",
]

for i, stage in enumerate(blueprint["stages"]):
    stage["notebook_code"] = ask_agent(SYSTEM_PROMPT, STAGE_PROMPTS[i])
    print(f"✅ Stage {stage['stage_number']}: {stage['stage_name']} — {len(stage['notebook_code'])} chars")

# COMMAND ----------
# MAGIC %md ### Step 3 — Agent creates notebooks in Databricks

requests.post(f"{DATABRICKS_HOST}/api/2.0/workspace/mkdirs", headers=db, json={"path": NOTEBOOK_FOLDER}).raise_for_status()

notebook_paths = []
for s in blueprint["stages"]:
    path = f"{NOTEBOOK_FOLDER}/{s['stage_number']:02d}_{s['stage_name'].replace(' ', '_')}"
    requests.post(f"{DATABRICKS_HOST}/api/2.0/workspace/import", headers=db, json={
        "path": path, "language": "PYTHON", "format": "SOURCE", "overwrite": True,
        "content": base64.b64encode(s["notebook_code"].encode()).decode()
    }).raise_for_status()
    notebook_paths.append(path)
    print(f"✅ Created: {path}")

# COMMAND ----------
# MAGIC %md ### Step 4 — Agent creates & triggers the Workflow

for job in requests.get(f"{DATABRICKS_HOST}/api/2.1/jobs/list", headers=db, params={"name": JOB_NAME}).json().get("jobs", []):
    requests.post(f"{DATABRICKS_HOST}/api/2.1/jobs/delete", headers=db, json={"job_id": job["job_id"]})

tasks = [{"task_key": f"stage_{i+1:02d}", "notebook_task": {"notebook_path": p},
          **({"depends_on": [{"task_key": f"stage_{i:02d}"}]} if i > 0 else {})}
         for i, p in enumerate(notebook_paths)]

job_id = requests.post(f"{DATABRICKS_HOST}/api/2.1/jobs/create", headers=db, json={
    "name": JOB_NAME,
    "tasks": tasks   # no cluster spec = Databricks uses serverless automatically
}).json()["job_id"]

run_id = requests.post(f"{DATABRICKS_HOST}/api/2.1/jobs/run-now", headers=db, json={"job_id": job_id}).json()["run_id"]
print(f"🚀 Running — Job: {job_id} | Run: {run_id}")
print(f"   {DATABRICKS_HOST}/#job/{job_id}/run/{run_id}")

# COMMAND ----------
# MAGIC %md ### Step 5 — Poll until all tasks green

start = time.time()
while True:
    info   = requests.get(f"{DATABRICKS_HOST}/api/2.1/jobs/runs/get", headers=db, params={"run_id": run_id}).json()
    state  = info["state"]["life_cycle_state"]
    result = info["state"].get("result_state", "—")
    icons  = {"SUCCESS": "✅", "FAILED": "❌"}
    print(f"[{int(time.time()-start):>3}s] {state}/{result}  " +
          "  ".join(f"{icons.get(t['state'].get('result_state',''),'🔄')} {t['task_key']}" for t in info.get("tasks",[])))
    if state in {"TERMINATED", "SKIPPED", "INTERNAL_ERROR"}: break
    time.sleep(20)

if result != "SUCCESS":
    for t in info.get("tasks", []):
        t_result = t["state"].get("result_state", "")
        print(f"  {'❌' if t_result=='FAILED' else '✅' if t_result=='SUCCESS' else '⏭'} {t['task_key']}: {t_result}")
        if t_result == "FAILED":
            task_run = requests.get(f"{DATABRICKS_HOST}/api/2.1/jobs/runs/get-output",
                                    headers=db, params={"run_id": t["run_id"]}).json()
            err   = task_run.get("error") or task_run.get("notebook_output", {}).get("result", "")
            trace = task_run.get("error_trace", "")
            print(f"   Error  : {err[:500]}")
            print(f"   Trace  : {trace[:800]}")
            print(f"   Raw    : {str(task_run)[:300]}")
    raise RuntimeError(f"Pipeline failed — open notebook in Databricks to see full error")
print("🎉 ALL TASKS GREEN!")

# COMMAND ----------
# MAGIC %md ### Step 6 — Results

# Find which catalog the pipeline wrote tables to, then display
_catalogs = [r[0] for r in spark.sql("SHOW CATALOGS").collect()]
_found_catalog = None
for _cat in ["hive_metastore", "main"] + [c for c in _catalogs if c not in ("hive_metastore","main")]:
    try:
        spark.sql(f"USE {_cat}.default")
        spark.table("results_summary_stats")  # probe
        _found_catalog = _cat
        print(f"✅ Found results tables in catalog: {_cat}.default")
        break
    except Exception:
        continue

if _found_catalog is None:
    raise RuntimeError("Could not locate results tables — check stage 4 logs for the catalog used")

display(spark.table("results_summary_stats"))
display(spark.table("results_top_email_domains"))
display(spark.table("results_top_first_names"))
display(spark.table("results_top_area_codes"))

# COMMAND ----------
# MAGIC %md ### Step 7 — Save blueprint

spark.sql("DROP TABLE IF EXISTS agent_pipeline_blueprint")
spark.createDataFrame([Row(
    pipeline_name=blueprint["pipeline_name"], stage_count=len(blueprint["stages"]),
    job_id=str(job_id), run_id=str(run_id), blueprint_json=json.dumps(blueprint)
)]).write.format("delta").mode("overwrite").saveAsTable("agent_pipeline_blueprint")
print("🏁 DONE — Fully automated POC complete.")